# Corrosion Detection for Bridges - YOLO11 Training

**Optimized for Google Colab with GPU**

This notebook trains a YOLO11 model to detect 4 types of corrosion in bridges:
- `rebar_corrosion` - rust staining + spalling from corroded rebar
- `steel_corrosion` - surface rust on steel beams, girders, bolts
- `rust_stain` - brown/orange streaking on concrete (early stage)
- `exposed_rebar` - rebar visible through spalled concrete

**Features:**
- Fully resumable: Click 'Run All' after disconnect to resume training
- 8-hour training budget optimized
- Datasets cached to Google Drive
- GPU-optimized (Tesla T4 or better)

## Cell 1: Configuration

**Edit these settings before running:**

In [ ]:
# Configuration - MODIFY THESE
ROBOFLOW_API_KEY = "5Zmgm6NCmNWapNJMTOtk"  # Your Roboflow API key

# Training settings
EPOCHS = 150                    # Total epochs (8 hours budget on T4)
PATIENCE = 40                   # Early stopping patience
BATCH_SIZE = 16                 # Auto if None, or set specific (8-16 for T4)
IMAGE_SIZE = 640                # Training resolution

# Class mapping - Map dataset classes to unified classes
# This will be auto-populated after dataset analysis
# Format: {dataset_name: {original_class_id: target_class_name}}
CLASS_MAPPING = {
    # CODEBRIM classes (from XML: background, crack, spalling, efflorescence, corrosion, exposed_bars)
    'codebrim': {
        'corrosion': 'rust_stain',        # Rust staining on concrete
        'exposed_bars': 'exposed_rebar',  # Visible rebar
        'spalling': 'rebar_corrosion',    # Spalling indicates corroded rebar
        # 'crack', 'efflorescence', 'background' - ignored
    },
    # Roboflow datasets will be auto-mapped based on class names
    'auto_map': True  # Set False to manually configure
}

# Resume from checkpoint?
FORCE_RESTART = False  # Set True to start fresh even if checkpoint exists

print("✓ Configuration loaded")
print(f"Training: {EPOCHS} epochs, batch={BATCH_SIZE or 'auto'}, imgsz={IMAGE_SIZE}")

## Cell 2: Setup and GPU Check

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
import sys
from pathlib import Path

# Workspace paths
WORKSPACE = Path('/content/corrosion_workspace')
WORKSPACE.mkdir(exist_ok=True)
os.chdir(WORKSPACE)

# Google Drive paths for persistence
DRIVE_BASE = Path('/content/drive/MyDrive/Bridge_Corrosion_YOLO11')
DRIVE_DATASETS = DRIVE_BASE / 'datasets'
DRIVE_MODELS = DRIVE_BASE / 'models'
DRIVE_RUNS = DRIVE_BASE / 'runs'
MERGED_DATASET_PATH = DRIVE_BASE / 'merged_dataset'

# Create directories
for path in [DRIVE_BASE, DRIVE_DATASETS, DRIVE_MODELS, DRIVE_RUNS]:
    path.mkdir(parents=True, exist_ok=True)

print(f"✓ Drive mounted")
print(f"Workspace: {WORKSPACE}")
print(f"Drive base: {DRIVE_BASE}")

# Check GPU
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Set device
    DEVICE = '0'
else:
    print("WARNING: No GPU detected! Training will be very slow.")
    DEVICE = 'cpu'

# Check disk space
!df -h /content

## Cell 3: Install Dependencies

In [ ]:
# Install required packages
!pip install -q ultralytics roboflow

import yaml
import json
import shutil
import xml.etree.ElementTree as ET
from collections import defaultdict
from tqdm.notebook import tqdm
import urllib.request
import zipfile

from ultralytics import YOLO
from roboflow import Roboflow

print("✓ Dependencies installed")

## Cell 4: Download CODEBRIM Dataset

Downloads from Zenodo and caches to Google Drive.

In [ ]:
def download_codebrim():
    """Download and extract CODEBRIM dataset"""
    codebrim_dir = DRIVE_DATASETS / 'codebrim_original'
    zip_path = DRIVE_DATASETS / 'CODEBRIM_original_images.zip'
    
    # Check if already extracted
    if codebrim_dir.exists() and any(codebrim_dir.iterdir()):
        print(f"✓ CODEBRIM already downloaded")
        return codebrim_dir
    
    # Download if needed
    if not zip_path.exists():
        print("Downloading CODEBRIM from Zenodo (8.3 GB)...")
        print("This may take 15-30 minutes...")
        url = "https://zenodo.org/api/records/2620293/files/CODEBRIM_original_images.zip/content"
        urllib.request.urlretrieve(url, zip_path)
        print(f"✓ Downloaded to {zip_path}")
    
    # Extract
    print("Extracting CODEBRIM...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(codebrim_dir)
    print(f"✓ Extracted to {codebrim_dir}")
    
    return codebrim_dir

# Download
CODEBRIM_DIR = download_codebrim()
print(f"\nCODEBRIM location: {CODEBRIM_DIR}")

## Cell 5: Convert CODEBRIM to YOLO Format

In [ ]:
def convert_codebrim_to_yolo(codebrim_dir, output_dir, class_mapping):
    """Convert CODEBRIM XML annotations to YOLO format"""
    
    # Find images directory
    images_dir = None
    for subdir in codebrim_dir.rglob('images'):
        if subdir.is_dir():
            images_dir = subdir
            break
    
    if not images_dir:
        raise ValueError("Could not find images directory in CODEBRIM")
    
    # Find metadata XML files
    metadata_dir = codebrim_dir / 'metadata'
    defects_xml = metadata_dir / 'defects.xml'
    background_xml = metadata_dir / 'background.xml'
    
    # Parse defects.xml (main annotations)
    tree = ET.parse(defects_xml)
    root = tree.getroot()
    
    # CODEBRIM class names (indices)
    codebrim_classes = ['background', 'crack', 'spalling', 'efflorescence', 'corrosion', 'exposed_bars']
    
    # Target unified classes
    unified_classes = ['rebar_corrosion', 'steel_corrosion', 'rust_stain', 'exposed_rebar']
    
    output_dir = Path(output_dir)
    output_images = output_dir / 'images'
    output_labels = output_dir / 'labels'
    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)
    
    # Process each image entry in XML
    converted_count = 0
    skipped_count = 0
    
    for image in tqdm(root.findall('image'), desc="Converting CODEBRIM"):
        filename = image.get('filename')
        if not filename:
            continue
        
        # Find image file
        img_path = images_dir / filename
        if not img_path.exists():
            skipped_count += 1
            continue
        
        # Get image dimensions
        from PIL import Image
        with Image.open(img_path) as img:
            img_width, img_height = img.size
        
        # Copy image
        new_img_name = f"codebrim_{filename}"
        shutil.copy2(img_path, output_images / new_img_name)
        
        # Convert boxes
        yolo_lines = []
        for box in image.findall('box'):
            # Get class from parent defect
            defect = box.getparent()
            if defect is None:
                continue
            
            # Get class labels from defect element
            labels = {}
            for i, cls_name in enumerate(codebrim_classes):
                elem = defect.find(cls_name)
                if elem is not None and elem.text == '1':
                    labels[cls_name] = i
            
            # Map to unified classes
            for cls_name in labels:
                if cls_name in class_mapping:
                    target_class = class_mapping[cls_name]
                    if target_class in unified_classes:
                        class_id = unified_classes.index(target_class)
                        
                        # Get box coordinates
                        x_min = float(box.get('xtl', 0))
                        y_min = float(box.get('ytl', 0))
                        x_max = float(box.get('xbr', 0))
                        y_max = float(box.get('ybr', 0))
                        
                        # Convert to YOLO format (normalized center x, y, width, height)
                        x_center = ((x_min + x_max) / 2) / img_width
                        y_center = ((y_min + y_max) / 2) / img_height
                        width = (x_max - x_min) / img_width
                        height = (y_max - y_min) / img_height
                        
                        # Clamp to [0, 1]
                        x_center = max(0, min(1, x_center))
                        y_center = max(0, min(1, y_center))
                        width = max(0, min(1, width))
                        height = max(0, min(1, height))
                        
                        yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
        
        # Write label file
        label_path = output_labels / (new_img_name.rsplit('.', 1)[0] + '.txt')
        with open(label_path, 'w') as f:
            f.write('\n'.join(yolo_lines))
        
        converted_count += 1
    
    print(f"✓ Converted {converted_count} images")
    print(f"  Skipped: {skipped_count} (images not found)")
    
    return output_dir, converted_count

# Convert CODEBRIM
codebrim_yolo_dir = DRIVE_DATASETS / 'codebrim_yolo'
if not codebrim_yolo_dir.exists() or not any((codebrim_yolo_dir / 'images').glob('*.jpg')):
    codebrim_yolo_dir, num_codebrim = convert_codebrim_to_yolo(
        CODEBRIM_DIR,
        codebrim_yolo_dir,
        CLASS_MAPPING['codebrim']
    )
else:
    num_codebrim = len(list((codebrim_yolo_dir / 'images').glob('*.jpg')))
    print(f"✓ CODEBRIM already converted ({num_codebrim} images)")

print(f"CODEBRIM YOLO dataset: {codebrim_yolo_dir}")

## Cell 6: Download Roboflow Datasets

In [ ]:
def download_roboflow_dataset(api_key, workspace_project, version, dataset_name, output_dir):
    """Download a dataset from Roboflow"""
    try:
        rf = Roboflow(api_key=api_key)
        workspace, project = workspace_project.split('/')
        project_obj = rf.workspace(workspace).project(project)
        
        # Get version (use specified or latest)
        if version is None:
            versions = project_obj.versions()
            version = versions[-1].version if versions else 1
        
        dataset = project_obj.version(version).download("yolov8", location=str(output_dir))
        return True
    except Exception as e:
        print(f"Error downloading {dataset_name}: {e}")
        return False

# Dataset configurations
ROBOFLOW_DATASETS = [
    ('tcc-fi6cb/detection-corrosion', 1, 'detection_corrosion'),  # Instance seg
    ('sutan-muhamad-sadam-awal-oo6pb/corrosion-detection-ofnck', 1, 'corrosion_detection_ofnck'),  # Object det
]

roboflow_datasets = []

for workspace_project, version, dataset_name in ROBOFLOW_DATASETS:
    output_dir = DRIVE_DATASETS / dataset_name
    
    if output_dir.exists() and (output_dir / 'data.yaml').exists():
        print(f"✓ {dataset_name}: Already downloaded")
    else:
        print(f"\nDownloading {dataset_name}...")
        success = download_roboflow_dataset(
            ROBOFLOW_API_KEY, workspace_project, version, dataset_name, output_dir
        )
        if not success:
            print(f"  ✗ Failed to download {dataset_name}")
            continue
        print(f"  ✓ Downloaded")
    
    roboflow_datasets.append((dataset_name, output_dir))

print(f"\n✓ Downloaded {len(roboflow_datasets)} Roboflow datasets")

## Cell 7: Analyze and Map Classes

In [ ]:
def analyze_dataset(dataset_path):
    """Analyze a YOLO dataset"""
    yaml_path = Path(dataset_path) / 'data.yaml'
    if not yaml_path.exists():
        return None
    
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    classes = data.get('names', [])
    
    # Count images
    counts = {}
    for split in ['train', 'valid', 'test']:
        img_dir = Path(dataset_path) / split / 'images'
        if img_dir.exists():
            counts[split] = len(list(img_dir.glob('*.*')))
        else:
            counts[split] = 0
    
    return {
        'path': str(dataset_path),
        'classes': classes,
        'nc': len(classes),
        'counts': counts,
        'total': sum(counts.values())
    }

# Analyze all datasets
print("=== Dataset Analysis ===\n")

# CODEBRIM (already converted)
print("CODEBRIM (converted):")
print(f"  Classes: rebar_corrosion, steel_corrosion, rust_stain, exposed_rebar")
print(f"  Images: {num_codebrim}\n")

# Roboflow datasets
dataset_infos = []
for name, path in roboflow_datasets:
    info = analyze_dataset(path)
    if info:
        dataset_infos.append((name, info))
        print(f"{name}:")
        print(f"  Classes: {info['classes']}")
        print(f"  Train: {info['counts']['train']}, Val: {info['counts']['valid']}, Test: {info['counts']['test']}")

# Auto-generate class mapping for Roboflow datasets
UNIFIED_CLASSES = ['rebar_corrosion', 'steel_corrosion', 'rust_stain', 'exposed_rebar']

def auto_map_class(class_name):
    """Map class name to unified class"""
    class_lower = class_name.lower()
    
    # Steel corrosion indicators
    if any(x in class_lower for x in ['steel', 'metal', 'iron', 'beam', 'girder', 'bolt']):
        return 'steel_corrosion'
    
    # Rebar corrosion indicators
    if any(x in class_lower for x in ['rebar', 're-bar', 'bar', 'spall', 'spalling']):
        return 'rebar_corrosion'
    
    # Exposed rebar
    if any(x in class_lower for x in ['exposed', 'visible', 'bar exposed']):
        return 'exposed_rebar'
    
    # Rust stain (early stage on concrete)
    if any(x in class_lower for x in ['stain', 'streak', 'discoloration', 'brown', 'orange']):
        return 'rust_stain'
    
    # Default to steel_corrosion for general corrosion
    if any(x in class_lower for x in ['corrosion', 'rust', 'corroded']):
        return 'steel_corrosion'
    
    return None  # Skip unknown classes

# Build complete class mappings
print("\n=== Class Mapping ===")
print(f"Unified classes: {UNIFIED_CLASSES}\n")

all_mappings = {
    'codebrim': {cls: target for cls, target in CLASS_MAPPING['codebrim'].items()}
}

for name, info in dataset_infos:
    print(f"{name}:")
    mapping = {}
    for i, cls in enumerate(info['classes']):
        if CLASS_MAPPING.get('auto_map', True):
            target = auto_map_class(cls)
        else:
            target = CLASS_MAPPING.get(name, {}).get(cls, None)
        
        if target:
            mapping[i] = target
            print(f"  {i}: {cls} -> {target}")
        else:
            print(f"  {i}: {cls} -> SKIPPED")
    
    all_mappings[name] = mapping
    print()

print("✓ Class mapping complete")

## Cell 8: Merge Datasets into Unified 4-Class Dataset

In [ ]:
def merge_datasets():
    """Merge all datasets into unified 4-class dataset"""
    
    # Check if already merged
    if MERGED_DATASET_PATH.exists() and (MERGED_DATASET_PATH / 'data.yaml').exists():
        # Check if we want to force re-merge
        if not FORCE_RESTART:
            print(f"✓ Merged dataset already exists")
            return MERGED_DATASET_PATH
    
    print("Creating unified 4-class dataset...\n")
    
    # Create directory structure
    splits = ['train', 'valid', 'test']
    for split in splits:
        (MERGED_DATASET_PATH / split / 'images').mkdir(parents=True, exist_ok=True)
        (MERGED_DATASET_PATH / split / 'labels').mkdir(parents=True, exist_ok=True)
    
    image_counter = 0
    class_counts = defaultdict(int)
    
    # Process CODEBRIM
    print("Processing CODEBRIM...")
    codebrim_images = Path(codebrim_yolo_dir) / 'images'
    codebrim_labels = Path(codebrim_yolo_dir) / 'labels'
    
    # Split CODEBRIM: 70% train, 15% val, 15% test
    codebrim_files = sorted(list(codebrim_images.glob('*.jpg')))
    n_total = len(codebrim_files)
    n_train = int(n_total * 0.7)
    n_val = int(n_total * 0.15)
    
    import random
    random.seed(42)
    random.shuffle(codebrim_files)
    
    splits_codebrim = {
        'train': codebrim_files[:n_train],
        'valid': codebrim_files[n_train:n_train+n_val],
        'test': codebrim_files[n_train+n_val:]
    }
    
    for split, files in splits_codebrim.items():
        for img_file in tqdm(files, desc=f"CODEBRIM {split}", leave=False):
            # Copy image
            new_name = f"img_{image_counter:06d}.jpg"
            shutil.copy2(img_file, MERGED_DATASET_PATH / split / 'images' / new_name)
            
            # Copy and process labels
            lbl_file = codebrim_labels / (img_file.stem + '.txt')
            if lbl_file.exists():
                with open(lbl_file, 'r') as f:
                    lines = f.readlines()
                
                # Update class IDs (CODEBRIM already has correct IDs)
                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        class_id = int(parts[0])
                        if 0 <= class_id < len(UNIFIED_CLASSES):
                            new_lines.append(line)
                            class_counts[UNIFIED_CLASSES[class_id]] += 1
                
                with open(MERGED_DATASET_PATH / split / 'labels' / (new_name.rsplit('.', 1)[0] + '.txt'), 'w') as f:
                    f.writelines(new_lines)
            
            image_counter += 1
    
    # Process Roboflow datasets
    for dataset_name, dataset_path in roboflow_datasets:
        print(f"\nProcessing {dataset_name}...")
        
        info = analyze_dataset(dataset_path)
        if not info:
            continue
        
        mapping = all_mappings.get(dataset_name, {})
        
        for split in ['train', 'valid', 'test']:
            src_img_dir = Path(dataset_path) / split / 'images'
            src_lbl_dir = Path(dataset_path) / split / 'labels'
            
            if not src_img_dir.exists():
                continue
            
            img_files = list(src_img_dir.glob('*.*'))
            
            for img_file in tqdm(img_files, desc=f"{dataset_name} {split}", leave=False):
                new_name = f"img_{image_counter:06d}{img_file.suffix}"
                shutil.copy2(img_file, MERGED_DATASET_PATH / split / 'images' / new_name)
                
                # Process labels
                lbl_file = src_lbl_dir / (img_file.stem + '.txt')
                if lbl_file.exists():
                    with open(lbl_file, 'r') as f:
                        lines = f.readlines()
                    
                    new_lines = []
                    for line in lines:
                        parts = line.strip().split()
                        if not parts:
                            continue
                        
                        orig_class = int(parts[0])
                        target_class = mapping.get(orig_class)
                        
                        if target_class and target_class in UNIFIED_CLASSES:
                            new_class_id = UNIFIED_CLASSES.index(target_class)
                            new_line = f"{new_class_id} {' '.join(parts[1:])}\n"
                            new_lines.append(new_line)
                            class_counts[target_class] += 1
                    
                    with open(MERGED_DATASET_PATH / split / 'labels' / (new_name.rsplit('.', 1)[0] + '.txt'), 'w') as f:
                        f.writelines(new_lines)
                
                image_counter += 1
    
    # Create data.yaml
    data_yaml = {
        'path': str(MERGED_DATASET_PATH),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': len(UNIFIED_CLASSES),
        'names': UNIFIED_CLASSES,
        'description': 'Bridge corrosion detection with 4 classes for Freyssinet'
    }
    
    with open(MERGED_DATASET_PATH / 'data.yaml', 'w') as f:
        yaml.dump(data_yaml, f, default_flow_style=False)
    
    # Count final statistics
    print("\n=== Merged Dataset Statistics ===")
    for split in splits:
        n = len(list((MERGED_DATASET_PATH / split / 'images').glob('*.*')))
        print(f"  {split}: {n} images")
    
    print("\nClass distribution:")
    for cls in UNIFIED_CLASSES:
        print(f"  {cls}: {class_counts[cls]} instances")
    
    print(f"\n✓ Merged dataset created at {MERGED_DATASET_PATH}")
    return MERGED_DATASET_PATH

# Merge datasets
MERGED_PATH = merge_datasets()

## Cell 9: Training (Fully Resumable)

In [ ]:
# Training configuration
RUN_NAME = 'yolo11m_bridge_corrosion'
RUNS_DIR = DRIVE_RUNS

# Check for existing run
run_path = RUNS_DIR / RUN_NAME
last_checkpoint = run_path / 'weights' / 'last.pt'
best_checkpoint = run_path / 'weights' / 'best.pt'

print("=== Training Status ===")
print(f"Run directory: {run_path}")

if FORCE_RESTART:
    print("⚠ FORCE_RESTART enabled - starting fresh")
    if run_path.exists():
        import shutil
        shutil.rmtree(run_path)
    resume_training = False
elif last_checkpoint.exists():
    print(f"✓ Found checkpoint: {last_checkpoint}")
    print("→ Will resume training")
    resume_training = True
else:
    print("No checkpoint found - starting fresh")
    resume_training = False

# Load model
if resume_training:
    print(f"\nLoading checkpoint: {last_checkpoint}")
    model = YOLO(str(last_checkpoint))
else:
    print("\nLoading pretrained YOLO11m...")
    model = YOLO('yolo11m.pt')

print(f"\nModel loaded successfully")
print(f"Device: {DEVICE}")

In [ ]:
# Training arguments
training_args = {
    'data': str(MERGED_DATASET_PATH / 'data.yaml'),
    'epochs': EPOCHS,
    'imgsz': IMAGE_SIZE,
    'batch': BATCH_SIZE or 'auto',
    'device': DEVICE,
    'patience': PATIENCE,
    'save_period': 10,  # Save every 10 epochs
    'project': str(RUNS_DIR),
    'name': RUN_NAME,
    'exist_ok': True,
    'pretrained': True,
    'optimizer': 'auto',
    'lr0': 0.01,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'cos_lr': True,
    'cache': False,  # Set True if enough RAM
    # Augmentations
    'degrees': 15.0,
    'translate': 0.2,
    'scale': 0.5,
    'shear': 5.0,
    'flipud': 0.5,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'mosaic': 1.0,
    'mixup': 0.2,
    'close_mosaic': 10,
    # Validation
    'val': True,
    'plots': True,
    'verbose': True,
}

print("Training Configuration:")
for k, v in training_args.items():
    print(f"  {k}: {v}")

# Start training
print("\n" + "="*60)
if resume_training:
    print("RESUMING TRAINING")
    print(f"Continuing from epoch checkpoint...")
    results = model.train(resume=True, **{k: v for k, v in training_args.items() if k != 'epochs'})
else:
    print("STARTING FRESH TRAINING")
    print(f"Target: {EPOCHS} epochs (~{EPOCHS//10} hours on T4)")
    results = model.train(**training_args)

print("\n" + "="*60)
print("Training Complete!")
print("="*60)

## Cell 10: Validation on Test Set

In [ ]:
# Validate on test set
print("\nRunning validation on test set...")

# Load best model if available
if best_checkpoint.exists():
    print(f"Loading best model: {best_checkpoint}")
    model = YOLO(str(best_checkpoint))

metrics = model.val(
    data=str(MERGED_DATASET_PATH / 'data.yaml'),
    split='test',
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE or 16,
    device=DEVICE,
    verbose=True,
    plots=True
)

print("\n" + "="*60)
print("TEST SET RESULTS")
print("="*60)
print(f"mAP50: {metrics.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"mAP50-95: {metrics.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"Precision: {metrics.results_dict.get('metrics/precision(B)', 0):.4f}")
print(f"Recall: {metrics.results_dict.get('metrics/recall(B)', 0):.4f}")

# Per-class mAP
print("\nPer-class mAP50:")
for i, cls in enumerate(UNIFIED_CLASSES):
    ap = metrics.ap50[i] if i < len(metrics.ap50) else 0
    print(f"  {cls}: {ap:.4f}")

## Cell 11: Export Model

In [ ]:
# Export to various formats
print("\nExporting model...\n")

formats = ['onnx', 'torchscript']

for fmt in formats:
    try:
        print(f"Exporting to {fmt.upper()}...")
        path = model.export(format=fmt, imgsz=IMAGE_SIZE, dynamic=True)
        print(f"  ✓ {path}")
        
        # Copy to Drive
        if os.path.exists(path):
            dest = DRIVE_MODELS / f"{RUN_NAME}.{fmt}"
            shutil.copy2(path, dest)
            print(f"  ✓ Copied to Drive: {dest}")
    except Exception as e:
        print(f"  ✗ Failed: {e}")

# Copy best model to Drive
if best_checkpoint.exists():
    drive_best = DRIVE_MODELS / f"{RUN_NAME}_best.pt"
    shutil.copy2(best_checkpoint, drive_best)
    print(f"\n✓ Best model saved: {drive_best}")

## Cell 12: Visualization and Summary

In [ ]:
# Display training results
from IPython.display import Image, display
import matplotlib.pyplot as plt

results_dir = run_path

print("\n=== Training Results ===\n")

# Training curves
for img_name in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    img_path = results_dir / img_name
    if img_path.exists():
        print(f"\n{img_name}:")
        display(Image(filename=str(img_path), width=800))

# Final summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Model: YOLO11m")
print(f"Classes: {', '.join(UNIFIED_CLASSES)}")
print(f"Dataset: {MERGED_DATASET_PATH}")
print(f"Run: {run_path}")
print("\nBest model:")
print(f"  {drive_best}")
print(f"\nTo resume training, just run all cells again.")
print("The training will automatically continue from the last checkpoint.")
print("="*60)

## Cell 13: Inference Demo (Optional)

In [ ]:
# Test on a sample image from the test set
import random

test_images = list((MERGED_DATASET_PATH / 'test' / 'images').glob('*.jpg'))
if test_images:
    sample_img = random.choice(test_images)
    
    print(f"Running inference on: {sample_img.name}")
    
    results = model(sample_img, conf=0.25)
    
    # Display
    for r in results:
        im_array = r.plot()  # BGR numpy array
        display(Image(data=im_array, format='png', width=800))
        
        print(f"\nDetected {len(r.boxes)} objects:")
        for box in r.boxes:
            cls_id = int(box.cls)
            conf = float(box.conf)
            print(f"  {UNIFIED_CLASSES[cls_id]}: {conf:.2%}")
else:
    print("No test images found")

---

## Notes

**Resumability:**
- All data is cached to Google Drive (`MyDrive/Bridge_Corrosion_YOLO11/`)
- Training checkpoints are saved every 10 epochs
- If Colab disconnects, simply run all cells again
- Training will resume from the latest checkpoint automatically

**8-Hour Training Budget:**
- YOLO11m on ~7k images at 640px takes ~4-6 hours for 150 epochs on T4
- Adjust EPOCHS in Cell 1 if needed

**Class Mapping:**
- Review Cell 7 output to verify class mappings are correct
- Edit `CLASS_MAPPING` in Cell 1 to customize

**Datasets:**
- CODEBRIM: Real bridge images (best for your use case)
- Roboflow: Additional diversity for generalization